# SecureLog Engine — Rapport synthétique

**Projet ING3 — Python 3.10+**  
**Membres du trinôme : Lukas MAUFFRÉ, Sacha BEHLOULI-DROLLÉE, Jeanne TONDA**

Nous avons choisi d'analyser les logs fictifs d'une salle informatique. Le
programme lit chaque ligne, vérifie son contenu puis met à jour les statistiques.
Nos trois règles recherchent les échecs de connexion consécutifs, les IP de la
blacklist et les événements critiques.

Avec nos fichiers de démonstration, nous obtenons **22 événements valides**,
**3 lignes rejetées** et **2 détections de force brute**. Le code principal se
trouve dans les fichiers `.py`. Ce Notebook sert à montrer les résultats.

## Architecture et choix

`main.py → ingestion → processing (filtrage, parser, normalisation) → analyzer + rules → résultats + audit`

- `ingestion.py` lit les fichiers et l'entrée standard avec `yield` et `yield from`.
- `parser.py` transforme les lignes en dictionnaires et vérifie leurs champs.
- `processing.py` utilise `filter`, `lambda`, `map` et `match-case`.
- `analyzer.py` calcule les compteurs et les ensembles d'IP.
- `rules.py` contient les règles de sécurité et la détection de force brute.
- `audit.py` utilise `globals()` et `locals()` pour décrire les variables.

Nous avons séparé les étapes pour éviter d'avoir tout le code dans `main.py`.
Le moteur utilise la bibliothèque standard. Les sorties du notebook sont sauvegardées et lisibles sur GitHub.
Les tests réexécutent ses cellules avec Python seul ; aucun paquet Jupyter
n'est nécessaire pour vérifier les résultats.

In [1]:
from pathlib import Path
import json
import sys
import tempfile

HERE = Path.cwd()
ROOT = HERE.parent if HERE.name == "report" else HERE
if not (ROOT / "main.py").is_file():
    raise RuntimeError("Ouvrir le notebook depuis la racine du projet ou report/")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from main import run

data = Path("data") if HERE == ROOT else Path("../data")
with tempfile.TemporaryDirectory() as directory:
    output = Path(directory)
    summary = run(
        [data / "auth.log", data / "network.log"],
        data / "blacklist.txt", output,
    )
    # Cette petite liste sert uniquement à illustrer les 15 alertes de démo.
    with (output / "alerts.jsonl").open(encoding="utf-8") as stream:
        alerts = [json.loads(line) for line in stream]
    audit = json.loads((output / "audit.json").read_text(encoding="utf-8"))

print("Entrées :", summary["input"])
print("Alertes :", summary["alerts"])

Entrées : {'read': 28, 'ignored': 3, 'invalid': 3, 'valid': 22}
Alertes : {'BLACKLIST': 12, 'BRUTE_FORCE': 2, 'CRITICAL': 1}


## Résultats par niveau

Les trois rejets sont intentionnels : mauvais nombre de champs, IP invalide et
JSON tronqué. Ils permettent de montrer la gestion d'exceptions. Les commentaires
et les lignes vides sont ignorés séparément. Une erreur de sévérité ERROR ne
signifie pas nécessairement une attaque : elle contribue aux statistiques.

In [2]:
for level, count in summary["by_level"].items():
    print(f"{level:8} {'█' * count} {count}")
print("IP actives :", len(summary["active_ips"]))
print("IP ayant généré ERROR ou CRITICAL :", summary["error_ips"])

CRITICAL █ 1
ERROR    ███████████ 11
INFO     █████ 5
WARNING  █████ 5
IP actives : 6
IP ayant généré ERROR ou CRITICAL : ['192.0.2.30', '198.51.100.24', '203.0.113.66']


## Ensembles et détections

La blacklist est un `frozenset` : les règles ne peuvent pas la modifier.
L'intersection désigne les IP actives blacklistées ; la différence les IP
actives non blacklistées. L'union rassemble les deux ensembles et la différence
symétrique contient les IP présentes dans un seul des deux ensembles.

In [3]:
for operation, addresses in summary["sets"].items():
    print(f"{operation} : {addresses}")
print("\nBrute force :")
for alert in alerts:
    if alert["rule"] == "BRUTE_FORCE":
        print(f"{alert['ip']} / {alert['user']} : {alert['message']}")

union : ['192.0.2.10', '192.0.2.20', '192.0.2.250', '192.0.2.30', '192.0.2.40', '198.51.100.24', '203.0.113.66']
intersection : ['198.51.100.24', '203.0.113.66']
difference : ['192.0.2.10', '192.0.2.20', '192.0.2.30', '192.0.2.40']
symmetric_difference : ['192.0.2.10', '192.0.2.20', '192.0.2.250', '192.0.2.30', '192.0.2.40']

Brute force :
203.0.113.66 / admin : 5 échecs consécutifs
198.51.100.24 / root : 5 échecs consécutifs


Les cinq échecs d'admin et de root déclenchent chacun une alerte. Bob a deux
échecs, un succès, puis deux échecs : aucune série de cinq n'existe. Les identités
`(IP, utilisateur)` sont séparées ; les événements réseau n'interrompent pas
les essais d'authentification. Chaque bloc disjoint de cinq échecs produit une
alerte : dix échecs produisent deux alertes.

Les fichiers sont lus dans l'ordre fourni, sans tri global. Les lignes rejetées
ne peuvent pas participer à la détection. Les alertes désignent des suspicions,
à confirmer en examinant les événements et leur provenance.

In [4]:
for name in ("blacklist", "counters", "histories", "analyzer"):
    print(name, audit["pipeline_locals"][name])
print("Module audité :", audit["audit_globals"]["AUDIT_VERSION"])

blacklist {'type': 'frozenset', 'size': 3}
counters {'type': 'dict', 'size': 4}
histories {'type': 'dict', 'size': 1}
analyzer {'type': 'Analyzer'}
Module audité : {'type': 'str', 'size': 3}


## Validation et consommation mémoire

Les **45 tests automatisés** passent dans des environnements virtuels
Python **3.10.22, 3.12.16, 3.14.10 et 3.15.2**. Ils couvrent le parsing,
les entrées incorrectes, la blacklist, les seuils, les ensembles, la
restauration des résultats en cas d'échec de publication et la CLI.
La publication est vérifiée dans 48 scénarios : échec de sauvegarde ou de
remplacement de chacun des trois fichiers, pour toutes les combinaisons
de résultats préexistants. Des tests simulent également une erreur d'export
et des octets UTF-8 invalides sur l'entrée standard, en modes normal et strict.

```bash
python -m unittest discover -s tests -v
```

La règle de force brute est comparée à un compteur de référence sur les
256 séquences possibles de huit tentatives, pour chacun des seuils de 1 à 5.
Un test vérifie qu'un succès libère immédiatement l'historique. Les nombres
JSON non finis, les chaînes Unicode invalides et les clés JSON répétées
sont refusés, même dans les objets imbriqués. Les tests vérifient aussi
que les variantes d'une même IP partagent la blacklist et l'historique.

Un test garde l'entrée standard ouverte et attend une alerte : il vérifie
ainsi son émission avant la fin du flux. Un autre réexécute les cellules de ce
notebook depuis la racine et depuis `report/`, puis compare leurs sorties
aux résultats sauvegardés, sans dépendance Jupyter. Le contrôle PEP 8 des
fichiers Python ne signale aucun écart.

Les mesures suivantes utilisent Python 3.12.16 et `tracemalloc`. La génération
du fichier a lieu avant la mesure. Chaque ligne reproduit un échec pour une
seule IP et un seul utilisateur ; le seuil vaut 5 et la blacklist est vide.
Les fichiers de résultats et les alertes sont réellement écrits sur disque.

| Lignes | Taille du fichier | Pic des allocations Python | Temps observé |
|---:|---:|---:|---:|
| 10 000 | 590 000 octets | 68 126 octets | 0,495 s |
| 100 000 | 5 900 000 octets | 67 655 octets | 4,685 s |
| 1 000 000 | 59 000 000 octets | 67 872 octets | 48,873 s |

Le million de lignes produit **200 000 alertes**. La taille du fichier est
multipliée par 100 entre les deux extrêmes, tandis que le pic reste voisin de
68 Ko pour cette population fixe. `tracemalloc` mesure les allocations
Python. Les temps dépendent de la machine.

Les agrégats et les tentatives en cours restent en mémoire. Leur taille
augmente avec le nombre d'IP, d'utilisateurs et de couples distincts. Un essai
avec 10 000 couples, chacun avec une IP et un utilisateur uniques, mesure
**17 265 535 octets**, soit environ 17,3 Mo.

Pour reproduire ces deux scénarios depuis la racine :

```bash
python -m tests.benchmark_memory --lines 10000 100000 1000000
python -m tests.benchmark_memory --lines 10000 --distinct
```


## Limites et conclusion

Le moteur traite les sources dans l'ordre fourni. Pour simuler un flux,
on peut lancer cette commande depuis la racine du projet.

```bash
cat data/auth.log data/network.log | python main.py --stream-alerts -
```

Les alertes arrivent à mesure que les lignes sont lues. Le résumé et l'audit
sont publiés à la fin d'une analyse réussie. Le seuil de force brute compte
les échecs consécutifs sans fenêtre temporelle.

## Source du cahier des charges

ÉNONCÉ DU PROJET FINAL : ENGINE DE TRAITEMENT ET D'ANALYSE DE FLUX DE DONNÉES
SÉCURISÉ EN TEMPS RÉEL, niveau ING3, document fourni par l'enseignant.